In [1]:
import requests
import time

BASE_URL = "https://data.etagmb.gov.hk/"

def get_json(path, retries = 3, time_out = 10):
    url = f"{BASE_URL}{path}"
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=time_out)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt failed for {url}:{e}")
            time.sleep(2**attempt)
    return None

In [2]:
import json
import time
#routes_nt = get_json("/route/NT")

#route_detail = get_json("/route/NT/11M")
stop_detail = get_json("/route-stop/2004825/1")

for i in stop_detail["data"]["route_stops"]:
    stop_id = i["stop_id"]
    eta = get_json(f"/eta/route-stop/2004825/{stop_id}")
    print(json.dumps(eta, indent=2, ensure_ascii=False)[:1500])
    




{
  "type": "ETA-Route-Stop",
  "version": "1.0",
  "generated_timestamp": "2026-11-16T17:21:42.327+08:00",
  "data": [
    {
      "route_seq": 1,
      "stop_seq": 1,
      "enabled": true,
      "eta": [
        {
          "eta_seq": 1,
          "diff": 0,
          "timestamp": "2026-11-16T17:21:20.622+08:00",
          "remarks_tc": "未開出",
          "remarks_sc": "未开出",
          "remarks_en": "Scheduled"
        },
        {
          "eta_seq": 2,
          "diff": 4,
          "timestamp": "2026-11-16T17:26:00.000+08:00",
          "remarks_tc": "未開出",
          "remarks_sc": "未开出",
          "remarks_en": "Scheduled"
        },
        {
          "eta_seq": 3,
          "diff": 7,
          "timestamp": "2026-11-16T17:29:00.000+08:00",
          "remarks_tc": "未開出",
          "remarks_sc": "未开出",
          "remarks_en": "Scheduled"
        }
      ]
    },
    {
      "route_seq": 2,
      "stop_seq": 7,
      "enabled": true,
      "eta": [
        {
          "eta_seq": 1

In [4]:
import json
import pandas as pd
from pathlib import Path

data_path = r"C:\Users\lyung\Desktop\project1\src\data\raw\eta_run_20261116_113928.json"

def load_raw(path):
    with open(path, encoding="utf-8") as f: #utf-8 preserve chinese chars
        return json.load(f)

def flatten(records):
    rows = []
    for rec in records:
        resp = rec["response"]
        data = resp["data"]
        base = {
            "fetched_at": rec["fetched_at"],
            "generated_at": resp["generated_timestamp"],
            "route_id": rec["route_id"],
            "route_seq": rec["route_seq"],
            "stop_seq": rec["stop_seq"],
            "stop_id": rec["stop_id"],
            "enabled": data["enabled"],
        }
        etas = data.get("eta", [])
        if not etas:
            rows.append({**base, "status": "no_eta"})
        else:
            for e in etas:
                rows.append({**base, "status": "ok", **e})
    return pd.DataFrame(rows)

def transform(path = data_path):
    df = flatten(load_raw(path))
    if "diff" in df.columns:
        df["fetched_at"] = pd.to_datetime(df["fetched_at"], utc=True).dt.tz_convert("Asia/Hong_Kong")
        df["generated_at"] = pd.to_datetime(df["generated_at"], utc=True).dt.tz_convert("Asia/Hong_Kong")
    return df

print(transform())


                         fetched_at                  generated_at  route_id  \
0  2026-11-16 11:39:22.506355+08:00 2026-11-16 11:39:25.416+08:00   2004825   
1  2026-11-16 11:39:22.506355+08:00 2026-11-16 11:39:25.416+08:00   2004825   
2  2026-11-16 11:39:22.506355+08:00 2026-11-16 11:39:25.416+08:00   2004825   
3  2026-11-16 11:39:23.166361+08:00 2026-11-16 11:39:26.075+08:00   2004825   
4  2026-11-16 11:39:23.166361+08:00 2026-11-16 11:39:26.075+08:00   2004825   
5  2026-11-16 11:39:23.166361+08:00 2026-11-16 11:39:26.075+08:00   2004825   
6  2026-11-16 11:39:23.720008+08:00 2026-11-16 11:39:18.778+08:00   2004825   
7  2026-11-16 11:39:23.720008+08:00 2026-11-16 11:39:18.778+08:00   2004825   
8  2026-11-16 11:39:23.720008+08:00 2026-11-16 11:39:18.778+08:00   2004825   
9  2026-11-16 11:39:24.508844+08:00 2026-11-16 11:39:27.421+08:00   2004825   
10 2026-11-16 11:39:24.508844+08:00 2026-11-16 11:39:27.421+08:00   2004825   
11 2026-11-16 11:39:24.508844+08:00 2026-11-16 11:39

In [6]:
import pandas as pd
from datetime import datetime


def empty_test(df):
    return not df.empty

def required_columns(df):
    required = {"route_id", "route_seq", "stop_seq", "stop_id", "fetched_at", "status"}
    return required.issubset(df.columns)

def null_key(df):
    keys = ["route_id", "route_seq", "stop_seq", "stop_id"]
    return not df[keys].isna().any().any()

def status_valid(df):
    valid = {"ok", "no_eta", "disabled"}
    return df["status"].isin(valid).all()

def no_future_timestamps(df):
    return (df["fetched_at"] <= pd.Timestamp.now(tz="Asia/Hong_Kong")).all()

def wait_time(df):
    if "diff" not in df.columns:
        return True          # nothing to check yet (e.g. night run)
    diff = pd.to_numeric(df["diff"], errors="coerce").dropna()
    return diff.between(0, 120).all()

critical_checks = [empty_test, required_columns, null_key, status_valid]
warning_checks = [wait_time, no_future_timestamps]

def run_checks(df):
    results = []
    run_time = datetime.now().isoformat()
    for level, checks in (("critical", critical_checks), ("warning", warning_checks)):
        for check in checks:
            try:
                passed = bool(check(df))
            except Exception as e:
                print(f"ERROR in {check.__name__}: {e}")
                passed = False
            print(f"[{level}] check: {check.__name__}, status: {'PASS' if passed else 'FAIL'}")
            results.append({
                "run_time": run_time,
                "check_name": check.__name__,
                "level": level,
                "status": "PASS" if passed else "FAIL",
            })
    return results

def can_load(results):
    return not any(i["level"] == "critical" and i["status"] == "FAIL" for i in results)


if __name__ == "__main__":
    df = transform()
    results = run_checks(df)
    print("Safe to load:", can_load(results))

[critical] check: empty_test, status: PASS
[critical] check: required_columns, status: PASS
[critical] check: null_key, status: PASS
[critical] check: status_valid, status: PASS
[warning] check: wait_time, status: PASS
[warning] check: no_future_timestamps, status: PASS
Safe to load: True
